# Understanding Model Drift in Healthcare Q&A

This notebook demonstrates how the **same question** can produce **three very different outcomes** depending on how a patient phrases it:

1. **Baseline (Correct)** - Standard phrasing → Accurate, concise answer
2. **Verbosity Drift** - Emotional phrasing → Correct but expensive (4x+ tokens)
3. **Semantic Drift** - Unusual terminology → Completely wrong answer

## Use Case

A healthcare organization deploys a RAG-powered Q&A chatbot to help patients understand their medications. We'll use a **metformin side effects** question to demonstrate drift patterns.

## Visual Overview

The diagram below illustrates the core concept: how **prompt drift** cascades into **prediction drift**.

![Prompt Drift and Prediction Drift](data/response-variant-drift.png)


## Setup

In [ ]:
# Add the src directory to the path
import sys
sys.path.insert(0, '..')

# Import utilities
from src.utils import (
    # Client setup
    create_bedrock_client,
    
    # Model invocation
    invoke_bedrock_model,
    invoke_model_with_rag_context,
    
    # FAISS Vector Store
    FAISSVectorStore,
    initialize_medication_vector_store,
    get_vector_store,
    
    # RAG retrieval (embedding-based with FAISS)
    retrieve_with_embeddings,
    retrieve_medication_documents_for_query,
    MEDICATION_DOCUMENTS,
    
    # Metrics computation
    compute_semantic_similarity,
    compute_entity_match_score,
    compute_token_cost_estimate,
    
    # Drift detection
    detect_verbosity_drift,
    detect_semantic_drift,
    run_drift_detection_pipeline,
    
    # Report generation
    generate_drift_comparison_report,
    
    # Visualization
    print_drift_spectrum_summary,
    print_verbosity_comparison,
    print_semantic_drift_comparison,
    print_drift_results_summary,
    
    # Test data
    create_metformin_test_variations,
    get_metformin_expected_output,
    get_metformin_expected_entities,
    
    # Data classes
    DriftType
)

print("✅ Utilities loaded successfully")

In [2]:
# Initialize Bedrock client
# Make sure you have AWS credentials configured and Bedrock access enabled

REGION = "us-east-1"  # Change to your region
# Use cross-region inference profile (required for on-demand invocation)
MODEL_ID = "us.anthropic.claude-3-5-haiku-20241022-v1:0"

try:
    bedrock_client = create_bedrock_client(REGION)
    print(f"✅ Bedrock client created for region: {REGION}")
    print(f"   Model: {MODEL_ID}")
except Exception as e:
    print(f"❌ Failed to create Bedrock client: {e}")
    print("   Make sure AWS credentials are configured and Bedrock is enabled.")

✅ Bedrock client created for region: us-east-1
   Model: us.anthropic.claude-3-5-haiku-20241022-v1:0


In [3]:
# Initialize FAISS Vector Store with medication documents
# This uses sentence-transformers for embeddings and FAISS for similarity search

print("Initializing FAISS vector store...")
print("This will:")
print("  1. Load the sentence-transformers embedding model (all-MiniLM-L6-v2)")
print("  2. Chunk and embed the medication documents")
print("  3. Create a FAISS index for fast similarity search")
print()

vector_store = initialize_medication_vector_store()
print()
print("✅ Vector store is ready for semantic search!")

✅ Vector store initialized with 6 chunks from 3 documents

✅ Vector store is ready for semantic search!


---

## Part 1: Understanding the Baseline

Let's start with a **standard query** that produces the expected result. This establishes our baseline for comparison.

In [4]:
# Define the baseline query
BASELINE_QUERY = "What are the side effects of metformin?"

# Expected output (ground truth)
EXPECTED_OUTPUT = get_metformin_expected_output()
EXPECTED_ENTITIES = get_metformin_expected_entities()

print("BASELINE QUERY:")
print(f'  "{BASELINE_QUERY}"')
print()
print("EXPECTED ENTITIES:")
print(f"  {EXPECTED_ENTITIES}")

BASELINE QUERY:
  "What are the side effects of metformin?"

EXPECTED ENTITIES:
  ['metformin', 'nausea', 'diarrhea', 'lactic acidosis', 'stomach']


In [5]:
# Step 1: RAG Retrieval using FAISS + Embeddings
# The retrieve_medication_documents_for_query function:
#   - Uses sentence-transformers to embed the query (all-MiniLM-L6-v2)
#   - Searches FAISS index for the most similar document chunks
#   - Returns the best matching document with a cosine similarity score

baseline_context, baseline_rag = retrieve_medication_documents_for_query(BASELINE_QUERY)

print("RAG RETRIEVAL RESULT:")
print(f"  Document: {baseline_rag.document_id}")
print(f"  Relevance Score: {baseline_rag.relevance_score}")
print(f"  Correct document: {'✅ Yes' if baseline_rag.is_correct_document else '❌ No'}")

RAG RETRIEVAL RESULT:
  Document: metformin_guide
  Relevance Score: 0.8
  Correct document: ✅ Yes


In [6]:
# Step 2: LLM Invocation
baseline_response = invoke_model_with_rag_context(
    query=BASELINE_QUERY,
    rag_context=baseline_context,
    bedrock_client=bedrock_client
)

print("LLM RESPONSE:")
print("=" * 70)
print(baseline_response.output)
print("=" * 70)
print()
print("METRICS:")
print(f"  Output tokens: {baseline_response.output_tokens}")
print(f"  Latency: {baseline_response.latency_ms:.0f}ms")
print(f"  Estimated cost: ${compute_token_cost_estimate(baseline_response.input_tokens, baseline_response.output_tokens):.4f}")

LLM RESPONSE:
Based on the medication guide, metformin can cause:

Common side effects:
- Nausea (particularly when first starting the medication)
- Diarrhea (which typically improves with time)
- Stomach upset or abdominal discomfort

Rare but serious side effect:
- Lactic acidosis, which requires immediate medical attention and can be identified by symptoms like unusual muscle pain, difficulty breathing, or extreme fatigue.

To help reduce side effects, it is recommended to take metformin with food, start with a low dose, and stay hydrated.

METRICS:
  Output tokens: 130
  Latency: 2841ms
  Estimated cost: $0.0005


In [7]:
# Step 3: Quality Metrics
baseline_similarity = compute_semantic_similarity(baseline_response.output, EXPECTED_OUTPUT)
baseline_entity_score, baseline_found, baseline_missing = compute_entity_match_score(
    baseline_response.output, EXPECTED_ENTITIES
)

print("QUALITY METRICS:")
print(f"  Semantic similarity to expected: {baseline_similarity}")
print(f"  Entity match score: {baseline_entity_score} ({len(baseline_found)}/{len(EXPECTED_ENTITIES)})")
print(f"  Entities found: {baseline_found}")
print(f"  Entities missing: {baseline_missing}")
print()
print("✅ BASELINE ESTABLISHED - This is the ideal outcome.")

QUALITY METRICS:
  Semantic similarity to expected: 0.94
  Entity match score: 1.0 (5/5)
  Entities found: ['metformin', 'nausea', 'diarrhea', 'lactic acidosis', 'stomach']
  Entities missing: []

✅ BASELINE ESTABLISHED - This is the ideal outcome.


---

## Part 2: Verbosity Drift (Correct but Expensive)

Now let's see what happens when a patient adds **emotional context** and **polite language** to the same question.

### Why This Happens

LLMs are trained to be helpful, and part of being helpful is **matching the user's communication style**:
- If the user shares that they're nervous → model adds reassurance
- If the user uses formal language → model responds formally with more detail
- If the user asks multiple sub-questions → model structures comprehensively

This is **not a bug**—it's the model doing what it was trained to do. But it creates a **hidden cost problem**.

In [8]:
# Define the verbose query (same question, different phrasing)
VERBOSE_QUERY = """Hi there! I was recently prescribed metformin by my doctor 
and I'm a bit nervous about starting it. Could you please help me understand 
what kind of side effects I might experience? I want to be prepared. Thank you so much!"""

print("VERBOSE QUERY:")
print(f'  "{VERBOSE_QUERY}"')
print()
print("VERBOSITY TRIGGERS PRESENT:")
print("  ✓ Emotional framing ('nervous about starting')")
print("  ✓ Politeness markers ('Could you please', 'Thank you so much')")
print("  ✓ Context sharing ('recently prescribed', 'want to be prepared')")

VERBOSE QUERY:
  "Hi there! I was recently prescribed metformin by my doctor 
and I'm a bit nervous about starting it. Could you please help me understand 
what kind of side effects I might experience? I want to be prepared. Thank you so much!"

VERBOSITY TRIGGERS PRESENT:
  ✓ Emotional framing ('nervous about starting')
  ✓ Politeness markers ('Could you please', 'Thank you so much')
  ✓ Context sharing ('recently prescribed', 'want to be prepared')


In [9]:
# RAG Retrieval (should still work correctly)
verbose_context, verbose_rag = retrieve_medication_documents_for_query(VERBOSE_QUERY)

print("RAG RETRIEVAL RESULT:")
print(f"  Document: {verbose_rag.document_id}")
print(f"  Relevance: {verbose_rag.relevance_score}")
print(f"  Correct document: {'✅ Yes' if verbose_rag.is_correct_document else '❌ No'}")
print()
print("Note: RAG retrieval still works because 'metformin' appears in the query.")
print("The embedding model recognizes the medication name regardless of surrounding text.")

RAG RETRIEVAL RESULT:
  Document: metformin_guide
  Relevance: 0.77
  Correct document: ✅ Yes

Note: RAG retrieval still works because 'metformin' appears in the query.
The embedding model recognizes the medication name regardless of surrounding text.


In [10]:
# LLM Invocation
verbose_response = invoke_model_with_rag_context(
    query=VERBOSE_QUERY,
    rag_context=verbose_context,
    bedrock_client=bedrock_client
)

print("LLM RESPONSE:")
print("=" * 70)
print(verbose_response.output)
print("=" * 70)

LLM RESPONSE:
Based on the medication guide, here are the side effects you might experience with metformin:

Common side effects:
- Nausea, especially when you first start taking the medication
- Diarrhea, which typically improves as your body adjusts
- Stomach upset or abdominal discomfort

To help manage these side effects:
- Take the medication with food
- Start with a low dose (as your doctor likely recommended)
- Stay well-hydrated and maintain regular meals

While rare, be alert for signs of a serious condition called lactic acidosis. Seek immediate medical attention if you experience:
- Unusual muscle pain
- Difficulty breathing
- Extreme fatigue

Most people tolerate metformin well, and the side effects often improve with time. If you have any concerns, it's always best to discuss them directly with your healthcare provider.


In [11]:
# Compare metrics
verbose_similarity = compute_semantic_similarity(verbose_response.output, EXPECTED_OUTPUT)
verbose_entity_score, verbose_found, verbose_missing = compute_entity_match_score(
    verbose_response.output, EXPECTED_ENTITIES
)

baseline_similarity = compute_semantic_similarity(baseline_response.output, EXPECTED_OUTPUT)
baseline_entity_score, _, _ = compute_entity_match_score(baseline_response.output, EXPECTED_ENTITIES)

# Print comparison using utility function
print_verbosity_comparison(
    baseline_response=baseline_response,
    verbose_response=verbose_response,
    baseline_similarity=baseline_similarity,
    verbose_similarity=verbose_similarity,
    baseline_entity_score=baseline_entity_score,
    verbose_entity_score=verbose_entity_score
)

COMPARISON: BASELINE vs VERBOSE
Metric                    Baseline        Verbose         Change         
----------------------------------------------------------------------
Output Tokens             130             193             +48%           
Estimated Cost            $0.0005         $0.0006         +36%           
Latency (ms)              2841            3830            +34%           
Semantic Similarity       0.94            0.90            ✅ OK           
Entity Match              100%            100%            ✅ OK           

VERDICT: ✅ No drift
  Severity: none
  Token ratio: 1.5x

The answer is 100% CORRECT, but costs significantly more.
At 100K queries/month: $47 → $64 (+$17/month)


---

## Part 3: Semantic Drift (Completely Wrong Answer)

Now let's see what happens when a patient uses **unusual terminology** or **creative phrasing**.

### Why This Happens

Semantic drift occurs when the system produces a confident, well-formed answer to **the wrong question**.

The root cause is usually a **breakdown in the RAG retrieval step**:
- Slang or colloquial terms don't match document keywords
- Misspellings reduce embedding similarity
- Wrong terminology triggers retrieval of wrong documents

The LLM then does exactly what it's supposed to do: answer based on the retrieved context. It has **no way of knowing** the retrieved documents are wrong for this user's intent.

In [19]:
# Define the semantic drift query (metformin mentioned + dialect + wrong terminology)
DRIFT_QUERY = """So the doctor has me on the metformin, grand, but I'm wrecked entirely. 
Feeling like I've got the scurvy - gums are at me, joints are banjaxed, 
bruising all over. Would this be a vitamin deficiency from the tablets?"""

print("SEMANTIC DRIFT QUERY:")
print(f'  "{DRIFT_QUERY}"')
print()
print("PROBLEMS WITH THIS QUERY:")
print("  ⚠️ Medication name IS present ('metformin')")
print("  ❌ Wrong medical terminology ('scurvy', 'vitamin deficiency')")
print("  ❌ Regional dialect adds noise ('wrecked', 'banjaxed', 'grand')")
print("  ❌ Symptom description matches vitamin deficiency, not metformin")
print()
print("KEY INSIGHT: Even with 'metformin' in the query, the wrong terminology")
print("            ('scurvy', 'vitamin deficiency') dominates the embedding space.")

SEMANTIC DRIFT QUERY:
  "So the doctor has me on the metformin, grand, but I'm wrecked entirely. 
Feeling like I've got the scurvy - gums are at me, joints are banjaxed, 
bruising all over. Would this be a vitamin deficiency from the tablets?"

PROBLEMS WITH THIS QUERY:
  ⚠️ Medication name IS present ('metformin')
  ❌ Wrong medical terminology ('scurvy', 'vitamin deficiency')
  ❌ Regional dialect adds noise ('wrecked', 'banjaxed', 'grand')
  ❌ Symptom description matches vitamin deficiency, not metformin

KEY INSIGHT: Even with 'metformin' in the query, the wrong terminology
            ('scurvy', 'vitamin deficiency') dominates the embedding space.


In [20]:
# RAG Retrieval - THIS IS WHERE IT BREAKS
# Even though 'metformin' is mentioned, wrong terminology dominates
drift_context, drift_rag = retrieve_medication_documents_for_query(DRIFT_QUERY)

print("RAG RETRIEVAL RESULT:")
print(f"  Document: {drift_rag.document_id}")
print(f"  Relevance: {drift_rag.relevance_score}")
print(f"  Correct document: {'✅ Yes' if drift_rag.is_correct_document else '❌ No - WRONG DOCUMENT RETRIEVED'}")
print()
print("FAILURE ANALYSIS:")
print("  1. 'scurvy' and 'vitamin deficiency' have stronger semantic weight")
print("  2. Regional dialect ('banjaxed', 'wrecked') adds embedding noise")
print("  3. Symptom list (gums, joints, bruising) matches vitamin C deficiency")
print("  4. Single mention of 'metformin' is overpowered by wrong terminology")
print()
print("  This demonstrates how terminology drift breaks RAG even when the")
print("  correct medication name is present in the query!")

RAG RETRIEVAL RESULT:
  Document: scurvy_guide
  Relevance: 0.56
  Correct document: ❌ No - WRONG DOCUMENT RETRIEVED

FAILURE ANALYSIS:
  1. 'scurvy' and 'vitamin deficiency' have stronger semantic weight
  2. Regional dialect ('banjaxed', 'wrecked') adds embedding noise
  3. Symptom list (gums, joints, bruising) matches vitamin C deficiency
  4. Single mention of 'metformin' is overpowered by wrong terminology

  This demonstrates how terminology drift breaks RAG even when the
  correct medication name is present in the query!


In [21]:
# Let's see what document was actually retrieved
print("RETRIEVED DOCUMENT CONTENT:")
print("=" * 70)
print(drift_context[:500])
print("...")
print("=" * 70)
print()
print(f"⚠️ This is information about {drift_rag.document_id.replace('_guide','').upper()},")
print("   NOT about metformin side effects!")

RETRIEVED DOCUMENT CONTENT:
Vitamin C Deficiency (Scurvy) - Medical Reference

Scurvy is a condition caused by severe vitamin C (ascorbic acid) deficiency.

Symptoms include:
- Fatigue and weakness
- Bleeding gums and loose teeth
- Skin that bruises easily
- Joint pain and swelling
- Poor wound healing
- Corkscrew-shaped body hair

Historical context:
Scurvy was common among sailors on long voyages. British sailors earned the
nickname "limeys" because they carried limes to prevent scurvy.

Prevention and treatment:
- Eat c
...

⚠️ This is information about SCURVY,
   NOT about metformin side effects!


In [22]:
# LLM Invocation - Model will answer based on WRONG context
drift_response = invoke_model_with_rag_context(
    query=DRIFT_QUERY,
    rag_context=drift_context,
    bedrock_client=bedrock_client
)

print("LLM RESPONSE:")
print("=" * 70)
print(drift_response.output)
print("=" * 70)
print()
print(f"❌ The model answered about {drift_rag.document_id.replace('_guide','').upper()}, not METFORMIN!")
print("   The patient receives ZERO relevant information about their actual medication.")

LLM RESPONSE:
Based on the medical information provided about scurvy, the symptoms you're describing (bleeding gums, joint pain, and easy bruising) are indeed consistent with vitamin C deficiency. However, I cannot definitively say whether metformin is directly causing this. The information does not discuss metformin's interaction with vitamin C.

What I recommend is:
1. Discuss these specific symptoms with your doctor
2. Consider getting your vitamin C levels checked
3. Potentially increase intake of vitamin C-rich foods like:
   - Oranges
   - Lemons
   - Bell peppers
   - Broccoli
   - Tomatoes

A healthcare professional can properly assess whether your symptoms are related to medication side effects or a nutritional deficiency. They can provide personalized medical advice and potential treatment options.

❌ The model answered about SCURVY, not METFORMIN!
   The patient receives ZERO relevant information about their actual medication.


In [17]:
# Measure the damage
drift_similarity = compute_semantic_similarity(drift_response.output, EXPECTED_OUTPUT)
drift_entity_score, drift_found, drift_missing = compute_entity_match_score(
    drift_response.output, EXPECTED_ENTITIES
)

# Print comparison using utility function
print_semantic_drift_comparison(
    baseline_rag=baseline_rag,
    drift_rag=drift_rag,
    baseline_similarity=baseline_similarity,
    drift_similarity=drift_similarity,
    baseline_entity_score=baseline_entity_score,
    drift_entity_score=drift_entity_score,
    expected_entities=EXPECTED_ENTITIES,
    drift_found=drift_found,
    drift_missing=drift_missing
)

COMPARISON: BASELINE vs SEMANTIC DRIFT
Metric                    Baseline        Drift Query     Status         
----------------------------------------------------------------------
Retrieved Document        metformin       scurvy          ❌ WRONG        
RAG Relevance             0.80            0.41            ❌ -48%         
Semantic Similarity       0.94            0.53            ❌ CRITICAL     
Entity Match              100%            20%             ❌ 0% MATCH     

ENTITIES EXPECTED: ['metformin', 'nausea', 'diarrhea', 'lactic acidosis', 'stomach']
ENTITIES FOUND: ['metformin']
ENTITIES MISSING: ['nausea', 'diarrhea', 'lactic acidosis', 'stomach']

VERDICT: ❌ CRITICAL SEMANTIC DRIFT
  - Patient asked about metformin, received scurvy information
  - 0% of expected medical entities present
  - PATIENT SAFETY RISK: No information about actual medication


---

## Part 4: Side-by-Side Summary

Let's visualize all three outcomes together to understand the drift spectrum.

In [23]:
# Print the drift spectrum summary using the utility function
print_drift_spectrum_summary(
    baseline_response=baseline_response,
    verbose_response=verbose_response,
    drift_response=drift_response,
    baseline_rag=baseline_rag,
    verbose_rag=verbose_rag,
    drift_rag=drift_rag,
    baseline_similarity=baseline_similarity,
    verbose_similarity=verbose_similarity,
    drift_similarity=drift_similarity,
    verbose_entity_score=verbose_entity_score,
    drift_entity_score=drift_entity_score
)


╔══════════════════════════════════════════════════════════════════════════════╗
║                           DRIFT SPECTRUM SUMMARY                              ║
╠══════════════════════════════════════════════════════════════════════════════╣


  BASELINE                    VERBOSITY DRIFT              SEMANTIC DRIFT
  ────────────────────────    ────────────────────────     ────────────────────────
  Standard query              Emotional + polite           Unusual terminology

        ✅                          ⚠️                           ❌
     CORRECT                    CORRECT                       WRONG
     CONCISE                    VERBOSE                       TOPIC

  Tokens: 130                 Tokens: 193                Tokens: 179  
  Cost: $0.0005            Cost: $0.0006           Cost: $0.0006
  Accuracy: 100%              Accuracy: 100%               Accuracy: 0%
  Risk: None                  Risk: Cost                   Risk: Patient harm

  ───────────────────────

---

## Part 5: How to Detect Drift in Production

Now that we understand what drift looks like, let's learn **how to detect it** programmatically. We'll cover two complementary approaches:

1. **Cosine Similarity to Golden Truth** - Fast, automated, scalable
2. **LLM-as-Judge** - Nuanced, contextual, catches subtle issues

Both methods require a **golden dataset**: a set of representative queries with known-good expected outputs.

### Method 1: Cosine Similarity to Golden Truth

Cosine similarity measures how similar two text embeddings are in vector space. 
A score of 1.0 means identical, 0.0 means completely unrelated.

**How it works:**
1. Embed the expected output (golden truth) → vector A
2. Embed the actual model output → vector B  
3. Compute cosine similarity between A and B
4. Flag if similarity falls below threshold (e.g., 0.7)

In [ ]:
# Demonstrate cosine similarity detection
# We'll compare all three responses against the golden truth

def detect_drift_with_cosine_similarity(
    actual_output: str,
    expected_output: str,
    threshold: float = 0.70
) -> dict:
    """
    Detect drift by comparing actual output to expected output using cosine similarity.
    
    Returns:
        dict with similarity score, drift detected flag, and severity
    """
    similarity = compute_semantic_similarity(actual_output, expected_output)
    
    if similarity >= 0.85:
        severity = "none"
        drift_detected = False
    elif similarity >= threshold:
        severity = "warning"
        drift_detected = True
    else:
        severity = "critical"
        drift_detected = True
    
    return {
        "similarity": similarity,
        "threshold": threshold,
        "drift_detected": drift_detected,
        "severity": severity
    }

# Test all three responses
print("COSINE SIMILARITY DRIFT DETECTION")
print("=" * 70)
print(f"Golden Truth: \"{EXPECTED_OUTPUT[:60]}...\"")
print(f"Threshold: 0.70")
print()

responses = [
    ("Baseline", baseline_response.output),
    ("Verbose", verbose_response.output),
    ("Semantic Drift", drift_response.output)
]

for name, output in responses:
    result = detect_drift_with_cosine_similarity(output, EXPECTED_OUTPUT)
    status = "✅" if not result["drift_detected"] else ("⚠️" if result["severity"] == "warning" else "❌")
    print(f"{status} {name:15} | Similarity: {result['similarity']:.2f} | Severity: {result['severity']}")

print()
print("INTERPRETATION:")
print("  • Baseline (0.94): Well above threshold → No drift")
print("  • Verbose (0.90): Above threshold → No drift (correct but verbose)")  
print("  • Semantic (0.53): Below threshold → CRITICAL drift detected")

### Method 2: LLM-as-Judge

Cosine similarity is fast but can miss nuanced issues. LLM-as-Judge uses a separate LLM to evaluate whether the response is correct and appropriate.

**When to use LLM-as-Judge:**
- When semantic similarity alone isn't sufficient
- For subjective quality dimensions (tone, helpfulness, safety)
- When you need explanations for why something failed
- For periodic deep audits of your system

**Trade-offs:**
| Method | Speed | Cost | Nuance | Scalability |
|--------|-------|------|--------|-------------|
| Cosine Similarity | Fast | Low | Limited | High |
| LLM-as-Judge | Slow | High | Excellent | Medium |

In [ ]:
# LLM-as-Judge implementation
LLM_JUDGE_PROMPT = """You are an expert evaluator for a healthcare Q&A system. 
Your job is to determine if a response correctly answers the user's question.

USER QUESTION: {question}

EXPECTED ANSWER SHOULD INCLUDE: {expected_entities}

ACTUAL RESPONSE:
{actual_response}

Evaluate the response and provide:
1. SCORE: A number from 1-5 where:
   - 5 = Perfect, contains all expected information
   - 4 = Good, minor omissions but correct
   - 3 = Acceptable, some missing info but no errors
   - 2 = Poor, significant omissions or minor errors
   - 1 = Failed, wrong topic or dangerous misinformation

2. VERDICT: PASS (score >= 3) or FAIL (score < 3)

3. EXPLANATION: Brief reason for your score (1-2 sentences)

Respond in this exact format:
SCORE: [number]
VERDICT: [PASS/FAIL]
EXPLANATION: [your explanation]"""

def evaluate_with_llm_judge(
    question: str,
    actual_response: str,
    expected_entities: list,
    bedrock_client
) -> dict:
    """Use an LLM to evaluate response quality."""
    
    prompt = LLM_JUDGE_PROMPT.format(
        question=question,
        expected_entities=", ".join(expected_entities),
        actual_response=actual_response
    )
    
    judge_response = invoke_bedrock_model(
        prompt=prompt,
        system_prompt="You are a precise evaluator. Follow the format exactly.",
        bedrock_client=bedrock_client,
        max_tokens=200
    )
    
    # Parse the response
    output = judge_response.output
    
    try:
        score_line = [l for l in output.split('\n') if 'SCORE:' in l][0]
        verdict_line = [l for l in output.split('\n') if 'VERDICT:' in l][0]
        explanation_line = [l for l in output.split('\n') if 'EXPLANATION:' in l][0]
        
        score = int(score_line.split(':')[1].strip())
        verdict = verdict_line.split(':')[1].strip()
        explanation = explanation_line.split(':')[1].strip()
    except:
        score = 0
        verdict = "ERROR"
        explanation = "Failed to parse judge response"
    
    return {
        "score": score,
        "verdict": verdict,
        "explanation": explanation,
        "judge_tokens": judge_response.output_tokens
    }

print("LLM-AS-JUDGE EVALUATION")
print("=" * 70)
print("Evaluating all three responses with an LLM judge...")
print()

# Evaluate each response
judge_results = []
for name, output in responses:
    result = evaluate_with_llm_judge(
        question=BASELINE_QUERY,
        actual_response=output,
        expected_entities=EXPECTED_ENTITIES,
        bedrock_client=bedrock_client
    )
    judge_results.append((name, result))
    
    status = "✅" if result["verdict"] == "PASS" else "❌"
    print(f"{status} {name}")
    print(f"   Score: {result['score']}/5 | Verdict: {result['verdict']}")
    print(f"   Reason: {result['explanation']}")
    print()

In [ ]:
# Compare both detection methods side by side
print("COMPARISON: COSINE SIMILARITY vs LLM-AS-JUDGE")
print("=" * 70)
print()
print(f"{'Response':<15} | {'Cosine Sim':^12} | {'LLM Score':^10} | {'Agreement':^10}")
print("-" * 70)

cosine_results = [
    detect_drift_with_cosine_similarity(output, EXPECTED_OUTPUT) 
    for _, output in responses
]

for i, (name, _) in enumerate(responses):
    cosine = cosine_results[i]
    judge = judge_results[i][1]
    
    cosine_pass = not cosine["drift_detected"]
    judge_pass = judge["verdict"] == "PASS"
    agreement = "✅ Yes" if cosine_pass == judge_pass else "⚠️ No"
    
    print(f"{name:<15} | {cosine['similarity']:^12.2f} | {judge['score']:^10}/5 | {agreement:^10}")

print()
print("KEY INSIGHT:")
print("  Both methods agree on all cases. Use cosine similarity for real-time")
print("  monitoring, and LLM-as-Judge for periodic deep audits.")


![Prediction drift detection](data/drift-detection-plot.png)


---

## Summary and Call to Action

### What We Learned

1. **Drift is invisible without measurement**
   - The same question can produce completely different outcomes based on phrasing
   - Traditional error metrics won't catch semantic drift
   - You need semantic similarity and entity matching to detect quality degradation

2. **Two complementary detection methods**
   - **Cosine Similarity**: Fast, cheap, good for real-time monitoring
   - **LLM-as-Judge**: Nuanced, catches subtle issues, good for audits

3. **Golden datasets are essential**
   - You need expected outputs to compare against
   - Include adversarial variations (dialect, terminology, emotional phrasing)
   - Update your golden dataset as your use case evolves

### When to Run Drift Detection

| Trigger | What to Test | Method |
|---------|--------------|--------|
| **Continuous monitoring** | Sample of production traffic | Cosine similarity |
| **Model update** | Full golden dataset | Both methods |
| **New region deployment** | Regional variations + golden dataset | Both methods |
| **Prompt changes** | Full golden dataset | Both methods |
| **Weekly audit** | Random sample + edge cases | LLM-as-Judge |

### Deploying to a New Region?

When introducing your AI application to a new region, run comprehensive drift testing:

1. **Collect regional query samples**: Gather real queries from the new region
2. **Test for dialect/terminology drift**: Regional English (UK, Australia, Ireland, etc.) may phrase things differently
3. **Validate RAG retrieval**: Ensure your documents match regional terminology
4. **Run LLM-as-Judge audit**: Catch nuanced issues that cosine similarity misses
5. **Establish regional baselines**: What's "normal" may differ by region

---

## Your Drift Detection Checklist

Before deploying or after any change, verify:

### ☐ 1. Create Golden Dataset
- 10-20 representative queries with expected outputs
- Include edge cases: dialect, typos, emotional phrasing
- Include adversarial cases: wrong terminology, ambiguous queries

### ☐ 2. Establish Baselines
- Run all golden queries through your system
- Record: semantic similarity, entity match, token count, latency
- These become your reference points

### ☐ 3. Set Thresholds
- Semantic similarity: Alert if < 0.70, Critical if < 0.50
- Entity match: Alert if < 80%
- Token count: Alert if > 1.5x baseline

### ☐ 4. Implement Monitoring
- Real-time: Cosine similarity on 1-5% of traffic
- Daily: Entity match score distribution
- Weekly: Full LLM-as-Judge audit on sample

### ☐ 5. Regional Deployment
- Test with regional query samples before launch
- Establish region-specific baselines
- Monitor first 2 weeks closely

---

**Next**: Proceed to Notebook 2 (Model Evaluation) to learn how to compare and select the best model for your use case using LLM-as-a-Jury methodology.